# MD Trajectory Analysis and GIF Export

This notebook loads `md_trajectories.npy` produced by `run_gcmc_hybrid.py`, inspects trajectory content, and exports a GIF for a selected MD move.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

ROOT = Path.cwd()

default_path = ROOT / "data/gcmc_runs/oa100.00deg_EAB1.000_mu-1.5000_kT1.0_r000/md_trajectories.npy"


OUTPUT_DIR = ROOT / "data/results/analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input trajectory file: {INPUT_NPY}")
print(f"Output directory:      {OUTPUT_DIR}")

In [ ]:
def load_md_trajectories(npy_path: Path):
    data = np.load(npy_path, allow_pickle=True, mmap_mode='r').item()

    if isinstance(data, dict) and "MD_trajectories" in data:
        raw = data["MD_trajectories"]
    elif isinstance(data, dict):
        raw = data
    else:
        raise ValueError("Unexpected file format: expected dict-like .npy payload.")

    traj_dict = {}
    for k, v in raw.items():
        traj_dict[int(k)] = np.asarray(v)

    return traj_dict

traj_dict = load_md_trajectories(INPUT_NPY)
move_ids = sorted(traj_dict.keys())

n_moves = len(move_ids)
frame_counts = np.array([traj_dict[i].shape[0] for i in move_ids], dtype=int)
particle_counts = np.array([traj_dict[i].shape[1] for i in move_ids], dtype=int)

print(f"Number of MD moves: {n_moves}")
print(f"Frames per move: min={frame_counts.min()}, max={frame_counts.max()}")
print(f"Particles per move: min={particle_counts.min()}, max={particle_counts.max()}")
print(f"First 10 move ids: {move_ids[:10]}")

In [ ]:
MOVE_ID = move_ids[0]  # Change to any available move id
traj = traj_dict[MOVE_ID]  # shape: (n_frames, N, 3)

xy = traj[:, :, :2]
xy_max = np.nanmax(np.abs(xy))
pad = 1.0
plot_lim = float(xy_max + pad)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

axes[0].scatter(traj[0, :, 0], traj[0, :, 1], s=25, color="tab:blue")
axes[0].set_title(f"Move {MOVE_ID}: first frame")
axes[0].set_xlim(-plot_lim, plot_lim)
axes[0].set_ylim(-plot_lim, plot_lim)
axes[0].set_aspect("equal")

axes[1].scatter(traj[-1, :, 0], traj[-1, :, 1], s=25, color="tab:orange")
axes[1].set_title(f"Move {MOVE_ID}: last frame")
axes[1].set_xlim(-plot_lim, plot_lim)
axes[1].set_ylim(-plot_lim, plot_lim)
axes[1].set_aspect("equal")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.show()

In [ ]:
def make_md_gif(traj, output_path: Path, fps: int = 12, point_size: int = 28):
    xy = traj[:, :, :2]
    xy_max = np.nanmax(np.abs(xy))
    pad = 1.0
    lim = float(xy_max + pad)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    scatter = ax.scatter([], [], s=point_size, c="tab:blue", alpha=0.9)
    title = ax.set_title("")

    def init():
        scatter.set_offsets(np.empty((0, 2)))
        title.set_text("")
        return scatter, title

    def update(i):
        pts = traj[i, :, :2]
        scatter.set_offsets(pts)
        title.set_text(f"Frame {i + 1}/{traj.shape[0]}")
        return scatter, title

    anim = FuncAnimation(
        fig,
        update,
        frames=traj.shape[0],
        init_func=init,
        blit=True,
        repeat=False,
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    anim.save(output_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return output_path

gif_path = OUTPUT_DIR / f"move_{MOVE_ID:05d}.gif"
saved_path = make_md_gif(traj, gif_path, fps=12)
print(f"Saved GIF: {saved_path}")
display(Image(filename=str(saved_path)))

In [ ]:
# Optional: export GIFs for the first few moves
N_EXPORT = 3
for move_id in move_ids[:N_EXPORT]:
    traj_i = traj_dict[move_id]
    out_i = OUTPUT_DIR / f"move_{move_id:05d}.gif"
    make_md_gif(traj_i, out_i, fps=12)
    print(f"Saved: {out_i}")